#### Deep Learning


Evaluate whether neural networks can improve RUL prediction compared with
the classical ML models from Phase 3.

The first neural-network model is an MLP.

If sequence-based modeling is justified, LSTM/GRU models will be evaluated
after the MLP baseline.

In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_dataset

train_df, test_df, rul_df = load_dataset("FD001")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (20631, 26)
Test shape: (13096, 26)


Load Dataset
Notebook starts from the same C-MAPSS training data used
in the previous phases.

This maintains consistency across the project.

In [2]:
RUL_CAP = 125

train_df["max_cycle"] = (
    train_df.groupby("unit")["cycle"].transform("max")
)

train_df["RUL_raw"] = (
    train_df["max_cycle"] - train_df["cycle"]
)

train_df["RUL"] = (
    train_df["RUL_raw"].clip(upper=RUL_CAP)
)

print("RUL minimum:", train_df["RUL"].min())
print("RUL maximum:", train_df["RUL"].max())
print("RUL missing:", train_df["RUL"].isna().sum())

RUL minimum: 0
RUL maximum: 125
RUL missing: 0


#### Reconstruct RUL Target

The Phase 1 RUL definition is reproduced so that the deep-learning
experiment uses the same prediction target as the classical ML models.

In [3]:
sensor_cols = [
    f"sensor_{i}"
    for i in range(1, 22)
]

diff_cols = [
    f"{sensor}_diff1"
    for sensor in sensor_cols
]

feature_cols = sensor_cols + diff_cols

print("Number of features:", len(feature_cols))

Number of features: 42


#### Define Model Features

The MLP uses the original sensor measurements together with the validated
one-cycle difference features from Phase 2.

This gives the neural network both current sensor information and recent
cycle-to-cycle change.

In [4]:
# Recreate Phase 2 difference features

train_df = train_df.sort_values(
    ["unit", "cycle"]
).reset_index(drop=True)

for sensor in sensor_cols:
    train_df[f"{sensor}_diff1"] = (
        train_df.groupby("unit")[sensor].diff(1)
    )

print("Difference features recreated:", len(diff_cols))

Difference features recreated: 21


In [5]:
print(train_df[diff_cols].head())

   sensor_1_diff1  sensor_2_diff1  sensor_3_diff1  sensor_4_diff1  \
0             NaN             NaN             NaN             NaN   
1             0.0            0.33            2.12            2.54   
2             0.0            0.20           -3.83            1.06   
3             0.0            0.00           -5.20           -2.33   
4             0.0            0.02            0.06            4.35   

   sensor_5_diff1  sensor_6_diff1  sensor_7_diff1  sensor_8_diff1  \
0             NaN             NaN             NaN             NaN   
1             0.0             0.0           -0.61           -0.02   
2             0.0             0.0            0.51            0.04   
3             0.0             0.0            0.19            0.03   
4             0.0             0.0           -0.45           -0.05   

   sensor_9_diff1  sensor_10_diff1  ...  sensor_12_diff1  sensor_13_diff1  \
0             NaN              NaN  ...              NaN              NaN   
1           -2.1

In [6]:
X = train_df[feature_cols].copy()

y = train_df["RUL"].copy()

X = X.fillna(0)

print("Missing feature values:", X.isna().sum().sum())

print("X shape:", X.shape)

print("y shape:", y.shape)

Missing feature values: 0
X shape: (20631, 42)
y shape: (20631,)


#### Handle Temporal Missing Values

The first cycle of each engine has no previous observation, so its
difference features are naturally missing.

These initial differences are represented as zero because no change from
a previous cycle is available.

In [7]:
from sklearn.model_selection import GroupShuffleSplit

groups = train_df["unit"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

groups_train = groups.iloc[train_idx]
groups_val = groups.iloc[val_idx]

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Training engines:", groups_train.nunique())
print("Validation engines:", groups_val.nunique())

Training rows: 16561
Validation rows: 4070
Training engines: 80
Validation engines: 20


#### Engine-Level Validation Split

The validation split is performed at the engine level.

This prevents observations from the same engine trajectory appearing in
both training and validation data.

The same principle was used during classical ML evaluation.

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

print("Training shape:", X_train_scaled.shape)
print("Validation shape:", X_val_scaled.shape)

Training shape: (16561, 42)
Validation shape: (4070, 42)


#### Feature Scaling

The input features are standardized using statistics learned only from
the training engines.

The same transformation is then applied to the validation data.

This prevents information from the validation set entering preprocessing.

In [9]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


#### Initialize Deep Learning Framework

TensorFlow/Keras is used to build the first neural-network model.

The first model is intentionally simple because it serves as the bridge
between classical tabular ML and sequence models.

In [10]:
model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         2,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,865 (19.00 KB)

 Trainable params: 4,865 (19.00 KB)

 Non-trainable params: 0 (0.00 B)

#### Build MLP Model

The MLP contains two hidden layers with ReLU activation.

The final layer contains one output because the task is regression:
predicting a single RUL value.

Dropout is included to reduce overfitting.

In [11]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 4474.5083 - mae: 54.5086 - val_loss: 1024.0726 - val_mae: 25.8698
Epoch 2/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 787.0236 - mae: 22.5981 - val_loss: 525.6861 - val_mae: 18.5744
Epoch 3/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 639.7070 - mae: 20.3331 - val_loss: 487.0020 - val_mae: 17.7211
Epoch 4/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 602.8735 - mae: 19.7165 - val_loss: 466.8196 - val_mae: 17.4227
Epoch 5/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 577.2777 - mae: 19.3604 - val_loss: 452.5187 - val_mae: 17.1525
Epoch 6/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 563.1996 - mae: 19.1018 - val_loss: 441.1127 - val_mae: 16.9273
Epoch 7/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 550.5823 - mae: 18.8971 - val_loss: 434.1117 - val_mae: 16.7734
Epoch 8/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 535.1624 - mae: 18.6591 - val_loss: 423.1691 - val_mae: 16.554

#### Train MLP

The MLP is trained using the engine-level training set.

Early stopping monitors validation loss and restores the best model
weights when validation performance stops improving.

In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_pred_mlp = model.predict(
    X_val_scaled,
    verbose=0
).ravel()

mlp_mae = mean_absolute_error(y_val, y_pred_mlp)

mlp_rmse = np.sqrt(
    mean_squared_error(y_val, y_pred_mlp)
)

print("MLP MAE:", mlp_mae)
print("MLP RMSE:", mlp_rmse)

MLP MAE: 11.66606616973877
MLP RMSE: 16.085087550396057


####  MLP Benchmark

The MLP provides the deep-learning baseline for comparison with
sequence-based models.

MLP performance:

- MAE: 11.73 cycles
- RMSE: 16.17 cycles

The MLP will be used as the reference model for the LSTM experiment.



In [13]:
print("Training epochs completed:", len(history.history["loss"]))

Training epochs completed: 86


In [14]:
print(
    "Best validation loss:",
    min(history.history["val_loss"])
)

Best validation loss: 258.7300720214844


#### Training Stability

Training history was inspected to confirm that the model trained
successfully and that early stopping selected the best validation state.

In [15]:
comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Gradient Boosting",
        "KNN",
        "Decision Tree",
        "Linear Regression",
        "Mean Baseline",
        "MLP"
    ],
    "MAE": [
        13.621029,
        13.976901,
        14.211369,
        14.947291,
        17.712183,
        36.965226,
        mlp_mae
    ],
    "RMSE": [
        18.799942,
        18.877456,
        20.248179,
        21.141506,
        21.634332,
        41.672689,
        mlp_rmse
    ]
})

comparison = comparison.sort_values("MAE").reset_index(drop=True)

comparison

,Model,MAE,RMSE
0,MLP,11.666066,16.085088
1,Random Forest,13.621029,18.799942
2,Gradient Boosting,13.976901,18.877456
3,KNN,14.211369,20.248179
4,Decision Tree,14.947291,21.141506
5,Linear Regression,17.712183,21.634332
6,Mean Baseline,36.965226,41.672689


#### Compare Classical ML and MLP

The MLP is compared with the Phase 3 classical models using the same
evaluation metrics.

The main question is whether the neural network improves upon the
Random Forest baseline.

In [16]:
best_model = comparison.iloc[0]

print("Best model:", best_model["Model"])
print("Best MAE:", best_model["MAE"])
print("Best RMSE:", best_model["RMSE"])

Best model: MLP
Best MAE: 11.66606616973877
Best RMSE: 16.085087550396057


#### Identify Best Model

The best model is selected based on the lowest validation MAE.

A neural network is only considered an improvement if it provides better
validation performance than the existing classical baseline.

In [17]:
rf_mae = 13.621029

mlp_improvement = (
    (rf_mae - mlp_mae) / rf_mae
) * 100

print(
    "MLP improvement over Random Forest (%):",
    mlp_improvement
)

MLP improvement over Random Forest (%): 14.352534087264852


#### MLP vs Random Forest

The MLP performance was compared directly with the Phase 3 Random Forest.

A positive improvement indicates that the MLP reduced MAE.

A negative improvement indicates that Random Forest remains the stronger
model.

#### Prepare Temporal Sequences

The MLP treats each engine cycle as an independent observation.

To investigate whether historical information improves RUL prediction,
a sequence-based LSTM model will be evaluated next.

A 20-cycle window will be used so that each sample contains recent engine
history.

In [18]:
sequence_features = sensor_cols

print("Number of sequence features:", len(sequence_features))
print(sequence_features)

Number of sequence features: 21
['sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']


In [19]:
X_seq_data = train_df[sequence_features].copy()
y_seq_data = y.copy()

X_seq_data = X_seq_data.fillna(0)

print("X shape:", X_seq_data.shape)
print("y shape:", y_seq_data.shape)
print("Missing values:", X_seq_data.isna().sum().sum())

X shape: (20631, 21)
y shape: (20631,)
Missing values: 0


#### Prepare Sequence Data

The sensor measurements and RUL target were separated.

Missing sensor values were replaced with zero to ensure a complete input
matrix before sequence construction.

In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_seq_data)

print("Scaled shape:", X_scaled.shape)
print("Mean:", X_scaled.mean())
print("Std:", X_scaled.std())

Scaled shape: (20631, 21)
Mean: 1.0624741914844358e-15
Std: 0.8451542547285166


#### Scale Sensor Features

The sensor features were standardized before sequence construction.

Standardization places the sensor variables on comparable numerical
scales, which helps stabilize neural-network training.

In [21]:
import numpy as np

WINDOW_SIZE = 20

X_sequences = []
y_sequences = []

for unit in train_df["unit"].unique():

    unit_indices = train_df.index[train_df["unit"] == unit].to_numpy()

    unit_X = X_scaled[unit_indices]
    unit_y = y_seq_data.iloc[unit_indices].to_numpy()

    for i in range(WINDOW_SIZE, len(unit_indices) + 1):

        X_sequences.append(unit_X[i-WINDOW_SIZE:i])
        y_sequences.append(unit_y[i-1])

X_sequences = np.array(X_sequences)
y_sequences = np.array(y_sequences)

print("Sequence shape:", X_sequences.shape)
print("Target shape:", y_sequences.shape)

Sequence shape: (18731, 20, 21)
Target shape: (18731,)


#### Create Temporal Sequences

A fixed window of 20 consecutive engine cycles was created for the LSTM.

Each training example therefore contains:

20 cycles × 21 sensor features

The target is the RUL associated with the final cycle in the sequence.

This allows the model to learn temporal patterns instead of treating each
cycle as an independent observation.

In [22]:
print("Number of sequences:", len(X_sequences))
print("Window size:", X_sequences.shape[1])
print("Number of features:", X_sequences.shape[2])
print("Targets:", len(y_sequences))

Number of sequences: 18731
Window size: 20
Number of features: 21
Targets: 18731


#### Validate Sequence Structure

The generated sequences were checked to ensure that every sample contains
20 consecutive cycles and all 21 sensor features.

The number of targets matches the number of generated sequences.

In [23]:
from sklearn.model_selection import GroupShuffleSplit

groups = []

for unit in train_df["unit"].unique():

    unit_indices = train_df.index[train_df["unit"] == unit].to_numpy()

    groups.extend([unit] * max(0, len(unit_indices) - WINDOW_SIZE + 1))

groups = np.array(groups)

print("Groups shape:", groups.shape)
print("Unique engines:", len(np.unique(groups)))

Groups shape: (18731,)
Unique engines: 100


In [24]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    gss.split(X_sequences, y_sequences, groups=groups)
)

X_lstm_train = X_sequences[train_idx]
X_lstm_val = X_sequences[val_idx]

y_lstm_train = y_sequences[train_idx]
y_lstm_val = y_sequences[val_idx]

print("Training sequences:", X_lstm_train.shape)
print("Validation sequences:", X_lstm_val.shape)

Training sequences: (15041, 20, 21)
Validation sequences: (3690, 20, 21)


#### Engine-Level Validation Split

The sequence data was divided using engine-level groups.

Sequences from the same engine are kept within the same split to prevent
information leakage between training and validation data.

This preserves the project-wide engine-level validation strategy.

In [25]:
lstm_model = keras.Sequential([
    layers.Input(
        shape=(WINDOW_SIZE, len(sequence_features))
    ),

    layers.LSTM(64),

    layers.Dense(32, activation="relu"),

    layers.Dense(1)
])

lstm_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        22,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,129 (94.25 KB)

 Trainable params: 24,129 (94.25 KB)

 Non-trainable params: 0 (0.00 B)

#### Build the LSTM Model

A simple LSTM architecture was created to learn temporal relationships
between consecutive engine cycles.

The model contains one LSTM layer followed by dense layers for RUL
regression.

In [26]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

#### Configure Early Stopping

Early stopping is used to stop training when validation performance stops
improving.

The best validation weights are restored after training.

In [27]:
history = lstm_model.fit(
    X_lstm_train,
    y_lstm_train,
    validation_data=(X_lstm_val, y_lstm_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
236/236 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - loss: 3508.4575 - mae: 46.7935 - val_loss: 481.9458 - val_mae: 18.4561
Epoch 2/100
236/236 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 360.2689 - mae: 15.2902 - val_loss: 252.7073 - val_mae: 12.7521
Epoch 3/100
236/236 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 284.4771 - mae: 12.9617 - val_loss: 218.1801 - val_mae: 10.6162
Epoch 4/100
236/236 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 251.8077 - mae: 11.9374 - val_loss: 196.7452 - val_mae: 10.5216
Epoch 5/100
236/236 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 224.2055 - mae: 11.1147 - val_loss: 240.4117 - val_mae: 11.2361
Epoch 6/100
236/236 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 220.4512 - mae: 10.9709 - val_loss: 192.8864 - val_mae: 9.9225
Epoch 7/100
236/236 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 202.6053 - mae: 10.4253 - val_loss: 198.6751 - val_mae: 9.5306
Epoch 8/100
236/236 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 191.6371 - mae: 10.0541 - val_loss: 211.7773 - val_mae: 9

#### Train the LSTM

The LSTM was trained using the engine-level training sequences.

Early stopping was used to prevent unnecessary training once validation
performance stopped improving.

In [28]:
print(
    "Training epochs completed:",
    len(history.history["loss"])
)

print(
    "Best validation loss:",
    min(history.history["val_loss"])
)

Training epochs completed: 16
Best validation loss: 192.8863525390625


#### Review LSTM Training

Training history was reviewed to determine how many epochs were completed
and the best validation loss achieved during training.

In [29]:
lstm_predictions = lstm_model.predict(
    X_lstm_val,
    verbose=0
).ravel()

print("Predictions:", lstm_predictions.shape)
print("Actual:", y_lstm_val.shape)

Predictions: (3690,)
Actual: (3690,)


#### Generate LSTM Predictions

The trained LSTM was used to predict RUL for the validation sequences.

The number of predictions matches the number of validation targets.

In [30]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

lstm_mae = mean_absolute_error(
    y_lstm_val,
    lstm_predictions
)

lstm_rmse = mean_squared_error(
    y_lstm_val,
    lstm_predictions
) ** 0.5

print("LSTM MAE:", lstm_mae)
print("LSTM RMSE:", lstm_rmse)

LSTM MAE: 9.922457695007324
LSTM RMSE: 13.88835257617956


#### Evaluate LSTM Performance

The LSTM was evaluated using MAE and RMSE.

MAE represents the average RUL prediction error in cycles, while RMSE gives
greater weight to larger prediction errors.

In [31]:
comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "MLP",
        "LSTM"
    ],
    "MAE": [
        13.621029,
        mlp_mae,
        lstm_mae
    ],
    "RMSE": [
        18.799942,
        mlp_rmse,
        lstm_rmse
    ]
})

comparison = comparison.sort_values("MAE")

print(comparison)

           Model        MAE       RMSE
2           LSTM   9.922458  13.888353
1            MLP  11.666066  16.085088
0  Random Forest  13.621029  18.799942


#### Compare Classical and Deep Learning Models

The LSTM was compared with the best classical model and the MLP using the
same MAE and RMSE metrics.

This comparison determines whether explicitly modeling temporal sequences
provides an advantage over tabular and non-sequential neural-network models.

In [32]:
lstm_improvement_over_mlp = (
    (mlp_mae - lstm_mae) / mlp_mae
) * 100

print(
    "LSTM improvement over MLP (%):",
    lstm_improvement_over_mlp
)

LSTM improvement over MLP (%): 14.945984784950767


#### Compare LSTM Against MLP

The LSTM's MAE was compared with the MLP benchmark.

A positive improvement indicates that sequence modeling reduced the RUL
prediction error.

A negative improvement indicates that the MLP performed better on the
current feature representation.

In [33]:

best_model = comparison.loc[
    comparison["MAE"].idxmin()
]

print("Best Phase 4 model:")
print(best_model)

Best Phase 4 model:
Model         LSTM
MAE       9.922458
RMSE     13.888353
Name: 2, dtype: object


In [34]:
# ============================================================
# SAVE FINAL LSTM MODEL
# ============================================================

from pathlib import Path

MODEL_DIR = Path("../models/deep_learning")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

LSTM_MODEL_PATH = MODEL_DIR / "lstm_model.keras"

lstm_model.save(LSTM_MODEL_PATH)

print("=" * 60)
print("FINAL LSTM MODEL SAVED")
print("=" * 60)
print("Path:", LSTM_MODEL_PATH.resolve())
print("Input shape:", lstm_model.input_shape)
print("Output shape:", lstm_model.output_shape)
print("MAE:", lstm_mae)
print("RMSE:", lstm_rmse)

FINAL LSTM MODEL SAVED
Path: C:\Users\User\Desktop\jet-engine-predictive-maintenance\models\deep_learning\lstm_model.keras
Input shape: (None, 20, 21)
Output shape: (None, 1)
MAE: 9.922457695007324
RMSE: 13.88835257617956


In [35]:
print("WINDOW_SIZE:", WINDOW_SIZE)
print("Number of sequence features:", len(sequence_features))
print("Sequence features:", sequence_features)
print("X_lstm_train:", X_lstm_train.shape)
print("X_lstm_val:", X_lstm_val.shape)

WINDOW_SIZE: 20
Number of sequence features: 21
Sequence features: ['sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']
X_lstm_train: (15041, 20, 21)
X_lstm_val: (3690, 20, 21)


#### GRU Model

The LSTM model achieved the best performance so far.

A GRU model is evaluated as a simpler alternative sequence architecture.
It uses the same temporal sequences and leakage-safe validation strategy
so that its performance can be compared fairly with the LSTM.

In [36]:
gru_model = keras.Sequential([
    layers.Input(
        shape=(WINDOW_SIZE, len(sequence_features))
    ),

    layers.GRU(64),

    layers.Dense(32, activation="relu"),

    layers.Dense(1)
])

gru_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 64)             │        16,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,817 (73.50 KB)

 Trainable params: 18,817 (73.50 KB)

 Non-trainable params: 0 (0.00 B)

In [37]:
gru_history = gru_model.fit(
    X_lstm_train,
    y_lstm_train,
    validation_data=(X_lstm_val, y_lstm_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
236/236 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - loss: 3633.1089 - mae: 48.6444 - val_loss: 550.5402 - val_mae: 20.3088
Epoch 2/100
236/236 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 444.0042 - mae: 17.5414 - val_loss: 316.4773 - val_mae: 14.4652
Epoch 3/100
236/236 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 372.9704 - mae: 15.1585 - val_loss: 263.9214 - val_mae: 12.8300
Epoch 4/100
236/236 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 314.8976 - mae: 13.8427 - val_loss: 250.1196 - val_mae: 12.2139
Epoch 5/100
236/236 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 287.3694 - mae: 13.0822 - val_loss: 215.5271 - val_mae: 10.7414
Epoch 6/100
236/236 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 259.7513 - mae: 12.1677 - val_loss: 201.4959 - val_mae: 10.5037
Epoch 7/100
236/236 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 242.5529 - mae: 11.5869 - val_loss: 228.6798 - val_mae: 10.8811
Epoch 8/100
236/236 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 228.1134 - mae: 11.1409 - val_loss: 200.5468 - val_mae:

In [38]:
gru_predictions = gru_model.predict(
    X_lstm_val,
    verbose=0
).ravel()

gru_mae = mean_absolute_error(
    y_lstm_val,
    gru_predictions
)

gru_rmse = mean_squared_error(
    y_lstm_val,
    gru_predictions
) ** 0.5

print("GRU MAE:", gru_mae)
print("GRU RMSE:", gru_rmse)

GRU MAE: 9.514503479003906
GRU RMSE: 13.619516371600318


In [39]:
print("GRU input shape:", gru_model.input_shape)
print("GRU output shape:", gru_model.output_shape)

GRU input shape: (None, 20, 21)
GRU output shape: (None, 1)


In [40]:
test_predictions = gru_model.predict(
    X_lstm_val,
    verbose=0
).flatten()

print("Prediction count:", len(test_predictions))
print("Prediction minimum:", test_predictions.min())
print("Prediction maximum:", test_predictions.max())
print("Prediction mean:", test_predictions.mean())

print("\nFirst 20 predictions:")
print(test_predictions[:20])

Prediction count: 3690
Prediction minimum: 1.2783917
Prediction maximum: 128.91336
Prediction mean: 81.759186

First 20 predictions:
[118.56603  118.419235 117.52333  119.32677  114.94938  119.453995
 120.8647   120.306465 121.69816  122.95575  125.63955  124.35593
 126.72119  123.92737  119.730705 122.472626 123.58807  125.858444
 125.717316 125.509026]


In [41]:
comparison = pd.DataFrame({
    "actual_RUL": y_lstm_val,
    "predicted_RUL": test_predictions
})

print(comparison.head(20))

print("\nPrediction statistics:")
print(comparison["predicted_RUL"].describe())

    actual_RUL  predicted_RUL
0          125     118.566032
1          125     118.419235
2          125     117.523331
3          125     119.326767
4          125     114.949379
5          125     119.453995
6          125     120.864700
7          125     120.306465
8          125     121.698158
9          125     122.955750
10         125     125.639549
11         125     124.355927
12         125     126.721191
13         125     123.927368
14         125     119.730705
15         125     122.472626
16         125     123.588074
17         125     125.858444
18         125     125.717316
19         125     125.509026

Prediction statistics:
count    3690.000000
mean       81.759186
std        39.347694
min         1.278392
25%        47.428734
50%        96.790039
75%       116.917915
max       128.913361
Name: predicted_RUL, dtype: float64


In [42]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(
    comparison["actual_RUL"],
    comparison["predicted_RUL"]
)

rmse = np.sqrt(
    mean_squared_error(
        comparison["actual_RUL"],
        comparison["predicted_RUL"]
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 9.514503479003906
RMSE: 13.619516371600318


#### Optional GRU Comparison

A GRU was evaluated using the same sequence representation and validation
strategy as the LSTM.

The purpose is to determine whether the simpler recurrent architecture
provides competitive performance.

In [43]:
comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "MLP",
        "LSTM",
        "GRU"
    ],
    "MAE": [
        13.621029,
        mlp_mae,
        lstm_mae,
        gru_mae
    ],
    "RMSE": [
        18.799942,
        mlp_rmse,
        lstm_rmse,
        gru_rmse
    ]
})

comparison = comparison.sort_values("MAE")

print(comparison)

           Model        MAE       RMSE
3            GRU   9.514503  13.619516
2           LSTM   9.922458  13.888353
1            MLP  11.666066  16.085088
0  Random Forest  13.621029  18.799942


In [44]:
best_model = comparison.loc[comparison["MAE"].idxmin()]

rf_mae = comparison.loc[
    comparison["Model"] == "Random Forest", "MAE"
].iloc[0]

mlp_mae = comparison.loc[
    comparison["Model"] == "MLP", "MAE"
].iloc[0]

best_mae = best_model["MAE"]

print("Best model:", best_model["Model"])
print("Best MAE:", best_model["MAE"])
print("Best RMSE:", best_model["RMSE"])

print(
    "GRU improvement over LSTM (%):",
    ((lstm_mae - best_mae) / lstm_mae) * 100
)

print(
    "GRU improvement over MLP (%):",
    ((mlp_mae - best_mae) / mlp_mae) * 100
)

print(
    "GRU improvement over Random Forest (%):",
    ((rf_mae - best_mae) / rf_mae) * 100
)

Best model: GRU
Best MAE: 9.514503479003906
Best RMSE: 13.619516371600318
GRU improvement over LSTM (%): 4.111423082294299
GRU improvement over MLP (%): 18.442915198920407
GRU improvement over Random Forest (%): 30.148423595574858


In [45]:
print("Validation sequences:", X_lstm_val.shape)
print("Validation targets:", y_lstm_val.shape)
print("GRU predictions:", gru_predictions.shape)

Validation sequences: (3690, 20, 21)
Validation targets: (3690,)
GRU predictions: (3690,)


In [46]:
gru_results = pd.DataFrame({
    "actual_RUL": y_lstm_val,
    "predicted_RUL": gru_predictions
})

print(gru_results.head())
print("Results shape:", gru_results.shape)

   actual_RUL  predicted_RUL
0         125     118.566032
1         125     118.419235
2         125     117.523331
3         125     119.326767
4         125     114.949379
Results shape: (3690, 2)


In [47]:
results = pd.DataFrame({
    "actual_RUL": y_lstm_val,
    "predicted_RUL": gru_predictions.flatten()
})

results["residual"] = (
    results["actual_RUL"] -
    results["predicted_RUL"]
)

results["absolute_error"] = results["residual"].abs()

print(results.head())
print("Results shape:", results.shape)

   actual_RUL  predicted_RUL   residual  absolute_error
0         125     118.566032   6.433968        6.433968
1         125     118.419235   6.580765        6.580765
2         125     117.523331   7.476669        7.476669
3         125     119.326767   5.673233        5.673233
4         125     114.949379  10.050621       10.050621
Results shape: (3690, 4)


In [48]:
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

results.to_csv(
    MODELS_DIR / "gru_model_results.csv",
    index=False
)

print("GRU results saved.")

GRU results saved.


In [49]:
from pathlib import Path
import numpy as np

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

gru_model.save(MODELS_DIR / "gru_model.keras")

np.save(MODELS_DIR / "X_lstm_val.npy", X_lstm_val)
np.save(MODELS_DIR / "y_lstm_val.npy", y_lstm_val)

print("GRU model and validation sequences saved.")
print("X_lstm_val:", X_lstm_val.shape)
print("y_lstm_val:", y_lstm_val.shape)

GRU model and validation sequences saved.
X_lstm_val: (3690, 20, 21)
y_lstm_val: (3690,)


In [50]:
from pathlib import Path

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

gru_model.save(
    MODELS_DIR / "gru_model.keras"
)

print("GRU model saved.")
print("Path:", MODELS_DIR / "gru_model.keras")

GRU model saved.
Path: ..\models\gru_model.keras


### Deep Learning Summary

Compare neural-network models with the classical ML models from Phase 3.

#### Models Evaluated

- MLP
- LSTM
- GRU
- Random Forest baseline

#### Results

| Model | MAE | RMSE |
|---|---:|---:|
| **GRU** | **9.2915** | **13.4996** |
| LSTM | 10.0091 | 13.8415 |
| MLP | 11.7329 | 16.1679 |
| Random Forest | 13.6210 | 18.7999 |

#### Key Findings

- MLP improved over Random Forest.
- LSTM further improved over MLP.
- GRU achieved the best validation performance.
- GRU achieved an MAE of approximately **9.29 cycles**.
- GRU achieved an RMSE of approximately **13.50 cycles**.

#### Conclusion

The results indicate that temporal deep-learning models outperform the classical ML baseline for this RUL prediction task.

The **GRU is the best-performing model in Phase 4** and will be used as the main model for subsequent analysis.

####  Output

- `notebooks/05_deep_learning.ipynb`
- `models/gru_model.keras`
- `models/gru_model_results.csv`

### Deep Learning Summary

Compare neural-network models with the classical ML models from Phase 3.

#### Models Evaluated

- MLP
- LSTM
- GRU
- Random Forest baseline

#### Results

| Model | MAE | RMSE |
|---|---:|---:|
| **GRU** | **9.2915** | **13.4996** |
| LSTM | 10.0091 | 13.8415 |
| MLP | 11.7329 | 16.1679 |
| Random Forest | 13.6210 | 18.7999 |

#### Key Findings

- MLP improved over Random Forest.
- LSTM further improved over MLP.
- GRU achieved the best validation performance.
- GRU achieved an MAE of approximately **9.29 cycles**.
- GRU achieved an RMSE of approximately **13.50 cycles**.

#### Conclusion

The results indicate that temporal deep-learning models outperform the classical ML baseline for this RUL prediction task.

The **GRU is the best-performing model in Phase 4** and will be used as the main model for subsequent analysis.

####  Output

- `notebooks/05_deep_learning.ipynb`
- `models/gru_model.keras`
- `models/gru_model_results.csv`

In [51]:
from pathlib import Path

GRU_MODEL_PATH = MODELS_DIR / "deep_learning" / "gru_model.keras"

print("GRU model path:")
print(GRU_MODEL_PATH)

print("Exists:", GRU_MODEL_PATH.exists())

GRU model path:
..\models\deep_learning\gru_model.keras
Exists: True


In [52]:
from tensorflow.keras.models import load_model

saved_gru = load_model(GRU_MODEL_PATH)

print("Loaded successfully")
print("Input shape:", saved_gru.input_shape)
print("Output shape:", saved_gru.output_shape)

Loaded successfully
Input shape: (None, 20, 21)
Output shape: (None, 1)


In [53]:
saved_predictions = saved_gru.predict(
    X_lstm_val,
    verbose=0
).flatten()

print("Min:", saved_predictions.min())
print("Max:", saved_predictions.max())
print("Mean:", saved_predictions.mean())

saved_mae = mean_absolute_error(
    y_lstm_val,
    saved_predictions
)

saved_rmse = np.sqrt(
    mean_squared_error(
        y_lstm_val,
        saved_predictions
    )
)

print("Saved GRU MAE:", saved_mae)
print("Saved GRU RMSE:", saved_rmse)

Min: -4.7967596
Max: 127.80386
Mean: 80.535484
Saved GRU MAE: 10.246235847473145
Saved GRU RMSE: 13.913873327426828


In [54]:
engine_id = 1

engine_df_test = test_df[
    test_df["unit"] == engine_id
].copy()

engine_df_test = engine_df_test.sort_values("cycle")

last_20 = engine_df_test[
    sequence_features
].tail(20)

print("Engine:", engine_id)
print("Rows:", len(last_20))
print("Shape:", last_20.shape)

X_test_engine = (
    last_20
    .to_numpy(dtype=np.float32)
    .reshape(1, 20, 21)
)

print("Model input shape:", X_test_engine.shape)
print("Min:", X_test_engine.min())
print("Max:", X_test_engine.max())
print("Mean:", X_test_engine.mean())

prediction = saved_gru.predict(
    X_test_engine,
    verbose=0
).flatten()[0]

print("GRU prediction:", prediction)

Engine: 1
Rows: 20
Shape: (20, 21)
Model input shape: (1, 20, 21)
Min: 0.03
Max: 9056.4
Mean: 1438.8062
GRU prediction: -22.329918


In [55]:
from tensorflow.keras.models import load_model
import numpy as np

saved_gru = load_model(
    r"C:\Users\User\Desktop\jet-engine-predictive-maintenance\models\deep_learning\gru_model.keras"
)

print("Saved model input:", saved_gru.input_shape)
print("Saved model output:", saved_gru.output_shape)

engine_id = 1

engine_df_test = test_df[
    test_df["unit"] == engine_id
].copy()

engine_df_test = engine_df_test.sort_values("cycle")

last_20 = engine_df_test[
    sequence_features
].tail(20)

X_test_engine = (
    last_20
    .to_numpy(dtype=np.float32)
    .reshape(1, 20, 21)
)

saved_prediction = saved_gru.predict(
    X_test_engine,
    verbose=0
).flatten()[0]

print("Saved GRU prediction:", saved_prediction)

Saved model input: (None, 20, 21)
Saved model output: (None, 1)
Saved GRU prediction: -22.329918


In [56]:
import numpy as np

print("GRU model:", gru_model)
print("Input shape:", gru_model.input_shape)
print("Output shape:", gru_model.output_shape)

print("\nTraining predictions:")

train_sample_predictions = gru_model.predict(
    X_lstm_train[:20],
    verbose=0
).flatten()

print(train_sample_predictions)

print("\nValidation predictions:")

val_sample_predictions = gru_model.predict(
    X_lstm_val[:20],
    verbose=0
).flatten()

print(val_sample_predictions)

print("\nValidation actual values:")

print(y_lstm_val[:20])

GRU model: <Sequential name=sequential_2, built=True>
Input shape: (None, 20, 21)
Output shape: (None, 1)

Training predictions:
[125.94294  127.64925  126.8642   124.531525 126.595795 125.04988
 127.87726  125.377594 127.05928  127.29468  126.967865 125.3557
 126.46034  125.22299  126.18666  124.52264  125.54666  125.90575
 124.505035 125.38771 ]

Validation predictions:
[118.56603  118.419235 117.52333  119.32677  114.94938  119.453995
 120.8647   120.306465 121.69816  122.95575  125.63955  124.35593
 126.72119  123.92737  119.730705 122.472626 123.58807  125.858444
 125.717316 125.50903 ]

Validation actual values:
[125 125 125 125 125 125 125 125 125 125 125 125 125 125 125 125 125 125
 125 125]
